# Task 4-3 + 4-4: Multi-label Image Classification Service

Train a **multi-label** classifier on the waste-objects dataset (6 classes),  
save weights, then serve via FastAPI.

**Tutorial refs (for PPT)**  
- https://machinelearningmastery.com/multi-label-classification-with-deep-learning/  
- https://www.analyticsvidhya.com/blog/2019/04/build-first-multi-label-image-classification-model-python/

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# Cell 1: Imports + device

import os, json, time
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from torchvision.models import MobileNet_V3_Small_Weights

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [ ]:
# Cell 2: Find dataset

DATASET_CANDIDATES = [
    os.environ.get("DATASET_DIR"),
    os.path.abspath("assignment-4/multi-label-objects-dataset"),
    "/content/drive/MyDrive/datasets/multi-label-objects-dataset",
]

DATA_DIR = None
for d in DATASET_CANDIDATES:
    if d and os.path.isdir(d):
        DATA_DIR = d
        break
if DATA_DIR is None:
    raise FileNotFoundError("Dataset not found.")

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VALID_DIR = os.path.join(DATA_DIR, "valid")
TEST_DIR  = os.path.join(DATA_DIR, "test")

CLASS_NAMES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
NUM_CLASSES = len(CLASS_NAMES)

print("Dataset:", DATA_DIR)
print("Classes:", CLASS_NAMES)

Dataset: /content/drive/MyDrive/datasets/multi-label-objects-dataset
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [12]:
# Cell 3: Build multi-label samples
#
# If the same image appears in multiple class folders it will have the
# same filename, so we group by filename to build multi-hot labels.
# (This is much faster than hashing every file.)

IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

def build_samples(split_dir):
    """Return list of (image_path, multi_hot_tensor)."""
    name_to_path = {}   # filename -> first path we found
    name_to_label = {}  # filename -> multi-hot vector

    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        cls_dir = os.path.join(split_dir, cls_name)
        for fname in os.listdir(cls_dir):
            if not fname.lower().endswith(IMG_EXTS):
                continue
            if fname not in name_to_path:
                name_to_path[fname] = os.path.join(cls_dir, fname)
                name_to_label[fname] = torch.zeros(NUM_CLASSES)
            name_to_label[fname][cls_idx] = 1.0

    return [(name_to_path[f], name_to_label[f]) for f in name_to_path]

train_samples = build_samples(TRAIN_DIR)
valid_samples = build_samples(VALID_DIR)
test_samples  = build_samples(TEST_DIR)

print(f"Train: {len(train_samples)} images")
print(f"Valid: {len(valid_samples)} images")
print(f"Test:  {len(test_samples)} images")

Train: 1766 images
Valid: 379 images
Test:  379 images


In [13]:
# Cell 4: Dataset + transforms + loaders

class MultiLabelDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        return self.transform(image), label


MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_loader = DataLoader(MultiLabelDataset(train_samples, train_transform), batch_size=32, shuffle=True,  num_workers=0)
valid_loader = DataLoader(MultiLabelDataset(valid_samples, eval_transform),  batch_size=32, shuffle=False, num_workers=0)
test_loader  = DataLoader(MultiLabelDataset(test_samples,  eval_transform),  batch_size=32, shuffle=False, num_workers=0)

print(f"Loaders ready — train: {len(train_loader)} batches, valid: {len(valid_loader)}, test: {len(test_loader)}")

Loaders ready — train: 56 batches, valid: 12, test: 12


In [14]:
# Cell 5: Model — MobileNetV3-Small (faster than Large, same idea)

model = models.mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, NUM_CLASSES)

for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"MobileNetV3-Small: {total/1e6:.1f}M params, {trainable:,} trainable")

MobileNetV3-Small: 1.5M params, 596,998 trainable


In [15]:
# Cell 6: Micro-F1 evaluation

@torch.no_grad()
def compute_f1_micro(model, loader, threshold=0.5):
    model.eval()
    tp = fp = fn = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        preds = (torch.sigmoid(model(images)) >= threshold).int()
        targets = labels.int()

        tp += ((preds == 1) & (targets == 1)).sum().item()
        fp += ((preds == 1) & (targets == 0)).sum().item()
        fn += ((preds == 0) & (targets == 1)).sum().item()

    precision = tp / max(tp + fp, 1)
    recall    = tp / max(tp + fn, 1)
    return 200.0 * precision * recall / max(precision + recall, 1e-12)  # 100 * 2pr/(p+r)

In [16]:
# Cell 7: Train

optimizer = torch.optim.Adam(
    (p for p in model.parameters() if p.requires_grad), lr=1e-3
)
criterion = nn.BCEWithLogitsLoss()

EPOCHS = 3
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    n = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        n += images.size(0)

    val_f1 = compute_f1_micro(model, valid_loader)
    print(f"Epoch {epoch}/{EPOCHS}  loss={total_loss/n:.4f}  val_f1={val_f1:.2f}%")

test_f1 = compute_f1_micro(model, test_loader)
print(f"\nTest F1 micro: {test_f1:.2f}%  ({time.time()-t0:.1f}s)")

Epoch 1/3  loss=0.3068  val_f1=70.64%
Epoch 2/3  loss=0.2007  val_f1=73.28%
Epoch 3/3  loss=0.1685  val_f1=77.84%

Test F1 micro: 77.40%  (1598.8s)


In [ ]:

artifacts_dir = os.path.join(DATA_DIR, "..", "artifacts")
os.makedirs(artifacts_dir, exist_ok=True)

weights_path = os.path.join(artifacts_dir, "multilabel_model.pt")
classes_path = os.path.join(artifacts_dir, "multilabel_classes.json")

torch.save(model.state_dict(), weights_path)
with open(classes_path, "w") as f:
    json.dump(CLASS_NAMES, f)

print("Saved:", os.path.abspath(weights_path))
print("Saved:", os.path.abspath(classes_path))


Saved: /content/drive/MyDrive/datasets/artifacts/multilabel_model.pt
Saved: /content/drive/MyDrive/datasets/artifacts/multilabel_classes.json
